# SAR Flood-Transfer: Cloud GPU Training Matrix

Runs the `sar-flood-transfer` project's Step 0.5 (oil-spill pretraining), Step 2 (45-run training matrix), Step 3 (27-run frozen-feature probing), and Step 4 (statistics + headline figure) on a Colab GPU.

**Before anything else:** `Runtime > Change runtime type > Hardware accelerator > GPU`, then run cells top to bottom.

**Persistence:** everything (dataset cache, checkpoints, results) is written to Google Drive, not the ephemeral Colab disk, so a session disconnect doesn't lose work. The training-matrix cells are resumable -- re-running them after a disconnect skips configs that already have a row in the results CSV instead of redoing them.

**One manual step required:** the Krestenitis oil-spill dataset (`Oil_Spill_Detection_Dataset`) isn't publicly re-hosted by this notebook -- you'll upload it to Drive once (see the cell below). Sen1Floods11 downloads automatically (public bucket, no auth).

## 1. Verify GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, (
    'No GPU detected. Go to Runtime > Change runtime type > select a GPU '
    '(T4 or better), then Runtime > Restart session and re-run this cell.'
)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

## 2. Mount Google Drive (persistent storage)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/sar-flood-transfer'
DATA_DIR = f'{DRIVE_ROOT}/data/sen1floods11'
KRESTENITIS_ZIP = f'{DRIVE_ROOT}/Oil_Spill_Detection_Dataset.zip'
KRESTENITIS_DIR = f'{DRIVE_ROOT}/Oil_Spill_Detection_Dataset'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
FIGURES_DIR = f'{DRIVE_ROOT}/figures'

for d in (DRIVE_ROOT, CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR):
    os.makedirs(d, exist_ok=True)
print('Persistent storage root:', DRIVE_ROOT)

## 3. Clone the repo

The repo is public, so this clones anonymously -- no token needed. (A token is still needed later, in the optional "push results back" cell, since writing requires authentication regardless of visibility.)

In [ ]:
import subprocess

REPO_PATH = '/content/research'
REPO_HOST_PATH = 'github.com/umahatokula/research.git'

if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', f'https://{REPO_HOST_PATH}', REPO_PATH], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_PATH, check=True)

PROJECT_DIR = f'{REPO_PATH}/sar-flood-transfer'
print('Repo ready at', PROJECT_DIR)

## 4. Install dependencies

(TensorFlow is skipped -- Colab's preinstalled GPU build is what we want; reinstalling from requirements-extra.txt could downgrade or break GPU support.)

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tifffile', 'imagecodecs', 'tqdm'], check=True)


def run(cmd, cwd=PROJECT_DIR):
    """Run a command, streaming output, raising on failure. Used for every training/eval step below."""
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {" ".join(cmd)}')

## 5. Sen1Floods11 data

Public bucket, no auth needed. Downloads to Drive so it's cached across sessions (~680MB, only downloaded once).

In [ ]:
run([sys.executable, 'src/data/download_sen1floods11.py', '--data-dir', DATA_DIR, '--max-workers', '12'])

## 6. Krestenitis oil-spill dataset (manual upload required)

On your own machine: zip `C:\Users\Umaha\Documents\phd\Oil_Spill_Detection_Dataset` and upload the zip to your Google Drive at `sar-flood-transfer/Oil_Spill_Detection_Dataset.zip` (i.e. inside the `sar-flood-transfer` folder Drive now has, from step 2). Then run this cell -- it unzips once and reuses the extracted copy on future runs.

In [ ]:
import zipfile

if not os.path.exists(KRESTENITIS_DIR):
    if not os.path.exists(KRESTENITIS_ZIP):
        raise FileNotFoundError(
            f'Upload Oil_Spill_Detection_Dataset.zip to {KRESTENITIS_ZIP} first, then re-run this cell.'
        )
    print('Unzipping (one-time)...')
    with zipfile.ZipFile(KRESTENITIS_ZIP) as zf:
        zf.extractall(DRIVE_ROOT)
print('Krestenitis dataset ready at', KRESTENITIS_DIR)

## 7. Step 0.5: oil-spill pretraining

Skips automatically if the checkpoint already exists on Drive from a previous session.

In [ ]:
OILSPILL_WEIGHTS = f'{CHECKPOINT_DIR}/oilspill_encoder.weights.h5'

if os.path.exists(OILSPILL_WEIGHTS):
    print('Checkpoint already exists, skipping pretraining:', OILSPILL_WEIGHTS)
else:
    run([
        sys.executable, '-m', 'src.training.pretrain_oilspill',
        '--dataset-root', KRESTENITIS_DIR,
        '--output-weights', OILSPILL_WEIGHTS,
        '--epochs', '30',
    ])

## 8. Step 2 smoke test

Per the handoff doc: confirm all 3 initializations work end to end at 10% labels / 1 seed before committing to the full 45-run matrix. Adjust `BATCH_SIZE` based on the GPU you were allocated (T4: 4 is safe; A100: try 8-16).

In [ ]:
BATCH_SIZE = 4
EPOCHS = 30
STEP2_CSV = f'{RESULTS_DIR}/step2_training_matrix.csv'
PER_CHIP_DIR = f'{RESULTS_DIR}/per_chip_metrics'

run([
    sys.executable, '-m', 'src.training.run_matrix',
    '--data-dir', DATA_DIR,
    '--output-csv', STEP2_CSV,
    '--per-chip-dir', PER_CHIP_DIR,
    '--oilspill-weights', OILSPILL_WEIGHTS,
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
])

## 9. Check smoke-test results before scaling up

In [ ]:
import pandas as pd

pd.read_csv(STEP2_CSV)

## 10. Full 45-run matrix

Long-running (hours). **Resumable**: if the session disconnects, just re-run this cell -- configs already in `step2_training_matrix.csv` are skipped, not redone. Only run this after checking the smoke-test results above look sane.

In [ ]:
run([
    sys.executable, '-m', 'src.training.run_matrix',
    '--data-dir', DATA_DIR,
    '--output-csv', STEP2_CSV,
    '--per-chip-dir', PER_CHIP_DIR,
    '--oilspill-weights', OILSPILL_WEIGHTS,
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--full',
])

## 11. Step 3: frozen-feature probing (27 runs)

Handoff doc: only run this after Step 2's results look sane. Also resumable.

In [ ]:
STEP3_CSV = f'{RESULTS_DIR}/step3_probe_matrix.csv'

run([
    sys.executable, '-m', 'src.training.frozen_probe',
    '--data-dir', DATA_DIR,
    '--output-csv', STEP3_CSV,
    '--oilspill-weights', OILSPILL_WEIGHTS,
    '--batch-size', str(BATCH_SIZE),
    '--epochs', '15',
])

## 12. Step 4: statistics + headline figure

In [ ]:
sys.path.insert(0, PROJECT_DIR)
from src.stats.significance import PRIMARY_LABEL_FRACTION, learning_curve_summary, primary_hypothesis_test
from src.figures.label_efficiency_curve import plot_label_efficiency_curve

curve_df = learning_curve_summary(STEP2_CSV)
primary = primary_hypothesis_test(PER_CHIP_DIR, label_fraction=PRIMARY_LABEL_FRACTION)
print('Primary comparison (oilspill vs imagenet, 10% labels):')
print(primary)

for metric in ('iou', 'f1'):
    out_path = f'{FIGURES_DIR}/label_efficiency_curve_{metric}.png'
    plot_label_efficiency_curve(curve_df, out_path, metric=metric, primary_result=primary)
    print('Saved', out_path)

In [ ]:
from IPython.display import Image, display

display(Image(filename=f'{FIGURES_DIR}/label_efficiency_curve_iou.png'))

## 13. Push results back to the repo (optional)

Copies the small results CSVs and figures (not raw data, not checkpoints -- those stay on Drive, same as they're gitignored locally) into the repo checkout and pushes.

In [ ]:
import getpass
import shutil

shutil.copytree(RESULTS_DIR, f'{PROJECT_DIR}/results', dirs_exist_ok=True)
shutil.copytree(FIGURES_DIR, f'{PROJECT_DIR}/figures', dirs_exist_ok=True)

subprocess.run(['git', 'add', 'sar-flood-transfer/results', 'sar-flood-transfer/figures'], cwd=REPO_PATH, check=True)
commit = subprocess.run(
    ['git', '-c', 'user.email=colab@example.com', '-c', 'user.name=Colab',
     'commit', '-m', 'Add sar-flood-transfer results from cloud run'],
    cwd=REPO_PATH,
)
if commit.returncode != 0:
    print('Nothing new to commit (or commit failed) -- check output above.')
else:
    push_token = getpass.getpass('GitHub token (needs write access), to push: ')
    subprocess.run(
        ['git', 'push', f'https://{push_token}@{REPO_HOST_PATH}', 'master'],
        cwd=REPO_PATH, check=True,
    )
    del push_token
    print('Pushed.')